### Setup

Environment setup and exploratory analysis

In [1]:
import sys
import platform
import numpy as np
import pandas as pd
import scipy
import scanpy as sc

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scanpy:", sc.__version__)

Python: 3.10.13 (main, Sep 19 2023, 15:54:50) [GCC 12.3.1 20230526]
Executable: /home/akash1/scgpt-cna/.venv-scgpt/bin/python
Platform: Linux-5.14.0-687.59.1.el9_8.x86_64-x86_64-Intel-R-_Xeon-R-_6972P-with-glibc2.37
NumPy: 1.26.4
Pandas: 2.3.3
SciPy: 1.12.0
Scanpy: 1.10.4


In [2]:
from pathlib import Path

DATA_DIR = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/benchmarking_spatial_scale/GDC-data/C3N-00148/C3N-00148-01/single_cell/dbe28c52-cdbf-477e-a728-9a61fd4a8139/qc_filtered_bc_feature_matrix"
)

assert DATA_DIR.exists()
print(DATA_DIR)

/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/benchmarking_spatial_scale/GDC-data/C3N-00148/C3N-00148-01/single_cell/dbe28c52-cdbf-477e-a728-9a61fd4a8139/qc_filtered_bc_feature_matrix


In [3]:
from pathlib import Path

import anndata as ad
import pandas as pd
from scipy import sparse
from scipy.io import mmread

DATA_DIR = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/"
    "benchmarking_spatial_scale/GDC-data/"
    "C3N-00148/C3N-00148-01/single_cell/"
    "dbe28c52-cdbf-477e-a728-9a61fd4a8139/"
    "qc_filtered_bc_feature_matrix"
)

features = pd.read_csv(
    DATA_DIR / "features.tsv",
    sep="\t",
    header=None,
    names=["gene_id", "gene_symbol", "feature_type"],
    dtype=str,
)

barcodes = pd.read_csv(
    DATA_DIR / "barcodes.tsv",
    sep="\t",
    header=None,
    names=["barcode"],
    dtype=str,
)

# Matrix Market file is genes × cells.
# AnnData convention is cells × genes, so transpose it.
counts = mmread(DATA_DIR / "matrix.mtx").T.tocsr()

assert sparse.issparse(counts)
assert counts.shape == (len(barcodes), len(features))

adata = ad.AnnData(X=counts)

adata.obs_names = barcodes["barcode"].to_numpy()
adata.var_names = features["gene_symbol"].to_numpy()

adata.var["gene_id"] = features["gene_id"].to_numpy()
adata.var["feature_type"] = features["feature_type"].to_numpy()

# Some gene symbols may be duplicated.
adata.var_names_make_unique()

/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/anndata/utils.py:261: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-4', 'SNORD116-5']
  warnings.warn(


In [4]:
print(adata)
print("Shape:", adata.shape)
print("Sparse:", sparse.issparse(adata.X))
print("Nonzero entries:", adata.X.nnz)
print(adata.var.head())
print(adata.obs.head())

AnnData object with n_obs × n_vars = 7598 × 60660
    var: 'gene_id', 'feature_type'
Shape: (7598, 60660)
Sparse: True
Nonzero entries: 12599909
                       gene_id     feature_type
DDX11L1      ENSG00000223972.5  Gene Expression
WASH7P       ENSG00000227232.5  Gene Expression
MIR6859-1    ENSG00000278267.1  Gene Expression
MIR1302-2HG  ENSG00000243485.5  Gene Expression
MIR1302-2    ENSG00000284332.1  Gene Expression
Empty DataFrame
Columns: []
Index: [AAACCCAAGAAGTGTT-1, AAACCCAAGCCTGTCG-1, AAACCCAAGCGCTGCT-1, AAACCCAAGTTCCGGC-1, AAACCCACAAGCGCTC-1]


In [5]:
df = pd.read_csv(DATA_DIR / "features.tsv", sep="\t")
features.head()

,gene_id,gene_symbol,feature_type
0,ENSG00000223972.5,DDX11L1,Gene Expression
1,ENSG00000227232.5,WASH7P,Gene Expression
2,ENSG00000278267.1,MIR6859-1,Gene Expression
3,ENSG00000243485.5,MIR1302-2HG,Gene Expression
4,ENSG00000284332.1,MIR1302-2,Gene Expression


In [6]:
from pathlib import Path
import pandas as pd

BASE = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/"
    "benchmarking_spatial_scale"
)
md = pd.read_csv(BASE / "metadata/sample_md.tsv", sep="\t")

diagnosis = "diagnoses.0.primary_diagnosis"

print("Metadata samples:", md["sample_id"].nunique())
print("Metadata cases:", md["case_id"].nunique())
assert md["sample_id"].is_unique

# Tissue / diagnosis composition, including numbers of distinct cases.
composition = (
    md.groupby(["primary_site", diagnosis], dropna=False)
    .agg(
        n_samples=("sample_id", "nunique"),
        n_cases=("case_id", "nunique"),
    )
    .reset_index()
)
display(composition)

# One row per person: which specimens and WGD labels are available?
cases = (
    md.groupby("case_id", sort=True)
    .agg(
        tissue=("primary_site", lambda x: ", ".join(sorted(set(x.dropna())))),
        diagnosis=(diagnosis, lambda x: "; ".join(sorted(set(x.dropna())))),
        n_samples=("sample_id", "nunique"),
        sample_ids=("sample_id", lambda x: ", ".join(sorted(x))),
        wgd_labels=("wgd_status", lambda x: ", ".join(sorted(set(x.dropna())))),
    )
    .reset_index()
)
display(cases)

# Check expected file locations without reading the large matrices.
availability = []

for _, row in md.iterrows():
    sample_dir = (
        BASE / "GDC-data" / row["case_id"] / row["sample_id"]
    )

    availability.append({
        "case_id": row["case_id"],
        "sample_id": row["sample_id"],
        "tissue": row["primary_site"],
        "diagnosis": row[diagnosis],
        "wgd_status": row["wgd_status"],
        "sample_directory_exists": sample_dir.is_dir(),
        "n_count_matrices": len(list(sample_dir.glob(
            "single_cell/*/qc_filtered_bc_feature_matrix/matrix.mtx*"
        ))),
        "n_wgs_segment_files": len(list(sample_dir.glob(
            "ground_truth/*/*.seg.txt"
        ))),
        "n_copykat_prediction_files": len(list(sample_dir.glob(
            "single_cell/*/copykatOUT/*prediction*.txt"
        ))),
    })

sample_overview = pd.DataFrame(availability)
display(sample_overview)

print("File counts are location checks, not validation of their contents.")

Metadata samples: 35
Metadata cases: 27


,primary_site,diagnoses.0.primary_diagnosis,n_samples,n_cases
0,Brain,Glioblastoma,16,16
1,Kidney,Angiomyolipoma,2,2
2,Kidney,Oncocytoma,2,2
3,Kidney,Papillary renal cell carcinoma,1,1
4,Kidney,"Renal cell carcinoma, NOS",14,6


,case_id,tissue,diagnosis,n_samples,sample_ids,wgd_labels
0,C3L-00359,Kidney,"Renal cell carcinoma, NOS",1,C3L-00359-01,WGD+
1,C3L-00606,Kidney,"Renal cell carcinoma, NOS",3,"C3L-00606-01, C3L-00606-02, C3L-00606-03",WGD+
2,C3L-01287,Kidney,"Renal cell carcinoma, NOS",2,"C3L-01287-01, C3L-01287-03","WGD+, WGD-"
3,C3L-01953,Kidney,Angiomyolipoma,1,C3L-01953-01,WGD-
4,C3L-02705,Brain,Glioblastoma,1,C3L-02705-71,WGD-
5,C3L-02858,Kidney,Angiomyolipoma,1,C3L-02858-01,WGD-
6,C3L-03405,Brain,Glioblastoma,1,C3L-03405-01,WGD+
7,C3L-03968,Brain,Glioblastoma,1,C3L-03968-01,WGD+
8,C3N-00148,Kidney,"Renal cell carcinoma, NOS",4,"C3N-00148-01, C3N-00148-02, C3N-00148-03, C3N-...","WGD+, WGD-"
9,C3N-00149,Kidney,"Renal cell carcinoma, NOS",3,"C3N-00149-02, C3N-00149-03, C3N-00149-04","WGD+, WGD-"


,case_id,sample_id,tissue,diagnosis,wgd_status,sample_directory_exists,n_count_matrices,n_wgs_segment_files,n_copykat_prediction_files
0,C3L-00359,C3L-00359-01,Kidney,"Renal cell carcinoma, NOS",WGD+,True,1,1,1
1,C3L-00606,C3L-00606-01,Kidney,"Renal cell carcinoma, NOS",WGD+,True,1,1,1
2,C3L-00606,C3L-00606-02,Kidney,"Renal cell carcinoma, NOS",WGD+,True,1,1,1
3,C3L-00606,C3L-00606-03,Kidney,"Renal cell carcinoma, NOS",WGD+,True,1,1,1
4,C3L-01287,C3L-01287-01,Kidney,"Renal cell carcinoma, NOS",WGD-,True,1,1,1
5,C3L-01287,C3L-01287-03,Kidney,"Renal cell carcinoma, NOS",WGD+,True,1,1,1
6,C3L-01953,C3L-01953-01,Kidney,Angiomyolipoma,WGD-,True,1,1,1
7,C3L-02705,C3L-02705-71,Brain,Glioblastoma,WGD-,True,1,1,1
8,C3L-02858,C3L-02858-01,Kidney,Angiomyolipoma,WGD-,True,1,1,1
9,C3L-03405,C3L-03405-01,Brain,Glioblastoma,WGD+,True,1,1,1


File counts are location checks, not validation of their contents.
